# Salinity preprocessing notebook

Workflow:

1. Configure paths, filters, and one baseline file
2. Convert `.xyz` → COG in `cogs_staging/salinity/` (**same filename**, `.tif` instead of `.xyz`)
   using the same grid / NaN empties as Food-Security `create_salinity_raster`
3. Compute salinity increase in `cogs_staging/salinity_increase/` (same filenames)
4. **Rename** COGs using the naming rule `{scenario}_{probability}_{year}.tif`
5. **Organize** renamed COGs into `stac_folder/` (same level as `cogs_staging`)
6. Sync `metadata_*.json` `NODATA` / `SPATIAL_RESOLUTION` from a sample COG


In [1]:
from pathlib import Path
import json
import re
import shutil

import numpy as np
import pandas as pd
import rasterio
from rasterio.shutil import copy as rio_copy
from rasterio.transform import from_origin


## 1) Configure paths and options

- `baseline_scenario` + `baseline_year`: the single COG subtracted when computing salinity increase
- `baseline_years`: all historical years that belong under `salinity/baseline/` and are **excluded** from increase


In [2]:
source_dir = Path(r"P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\projections_gridded")
output_root = Path(r"P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs")

probability = "p50"
allowed_suffixes = ["y", "sb2y", "sm2y", "sb2rb1y", "sm2rb1y", "sb2rb3y"]

# Selected baseline used for scenario - baseline (must exist after conversion)
baseline_scenario = "cc45y"
baseline_year = "2018"
# All historical years → salinity/baseline/; never used as increase targets
baseline_years = ["2014", "2015", "2016", "2018"]

# Match Food-Security create_salinity_raster: CRS + NaN empties.
# Resolution is derived from unique XYZ coordinates (not a fixed 2000 m).
source_crs = "EPSG:32648"
nodata_value = np.nan

config = {
    "source_dir": source_dir,
    "output_root": output_root,
    "probability": probability,
    "allowed_suffixes": allowed_suffixes,
    "baseline_scenario": baseline_scenario,
    "baseline_year": baseline_year,
    "baseline_years": baseline_years,
    "source_crs": source_crs,
    "nodata_value": nodata_value,
}

# Path variables only (folders are created when needed)
cogs_staging = output_root / "cogs_staging"
salinity_staging = cogs_staging / "salinity"
salinity_increase_staging = cogs_staging / "salinity_increase"

# Final STAC output (sibling folder of cogs_staging)
stac_folder = output_root / "stac_folder"

print("Source:", source_dir)
print("Output:", output_root)
print("COG staging:", cogs_staging)
print("STAC folder:", stac_folder)
print("Nodata:", nodata_value)
print("Subtract baseline:", f"*_{baseline_scenario}{baseline_year[-2:]}.xyz")
print("Baseline years (folder / no increase):", baseline_years)


Source: P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\projections_gridded
Output: P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs
COG staging: P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\cogs_staging
STAC folder: P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\stac_folder
Nodata: nan
Subtract baseline: *_cc45y18.xyz
Baseline years (folder / no increase): ['2014', '2015', '2016', '2018']


## 2) Helper functions


In [3]:
SHORT_NAME_RE = re.compile(r"^(p\d+)_(cc\d{2}[a-z0-9]+)(\d{2})\.(xyz|tif)$", re.IGNORECASE)
XYZ_FILTER_RE = re.compile(r"_(cc\d{2}[a-z0-9]+?)(\d+)\.(xyz|tif)$", re.IGNORECASE)
RENAMED_NAME_RE = re.compile(r"^(cc\d{2}[a-z0-9]+)_(p\d+)_(\d{4})\.tif$", re.IGNORECASE)


def parse_tif_metadata(file_path):
    """Read scenario, probability and year from xyz-style or renamed filenames."""
    name = file_path.name

    renamed = RENAMED_NAME_RE.match(name)
    if renamed:
        scenario = renamed.group(1).lower()
        return {
            "scenario": scenario,
            "probability": renamed.group(2).lower(),
            "year": renamed.group(3),
            "suffix": scenario[4:],
        }

    short = SHORT_NAME_RE.match(name)
    if short:
        scenario = short.group(2).lower()
        return {
            "scenario": scenario,
            "probability": short.group(1).lower(),
            "year": f"20{short.group(3)}",
            "suffix": scenario[4:],
        }

    match = XYZ_FILTER_RE.search(name.lower())
    if match:
        scenario = match.group(1).lower()
        return {
            "scenario": scenario,
            "probability": name.lower().split("_")[0],
            "year": f"20{match.group(2)}",
            "suffix": scenario[4:],
        }

    return None


def matches_criteria(file_path, config):
    """Return file metadata only when probability and scenario suffix match config."""
    parsed = parse_tif_metadata(file_path)
    if not parsed:
        return None
    if parsed["probability"] != config["probability"].lower():
        return None
    if parsed["suffix"] not in config["allowed_suffixes"]:
        return None
    return parsed


def convert_xyz_to_cog(xyz_path, out_tif_path, config):
    """Convert one xyz file to COG.

    Gridding matches Food-Security `create_salinity_raster` exactly:
    - resolution from unique x/y spacing
    - origin at (x.min, y.max)
    - empty cells = NaN
    - dtype from ``np.loadtxt`` (float64)
    - same row/col formulas and ``width + 1`` fill array (profile width = n_unique_x)
    """
    out_tif_path.parent.mkdir(parents=True, exist_ok=True)

    source_crs = config["source_crs"]
    nodata_value = config["nodata_value"]

    salinity = np.loadtxt(xyz_path)
    xcoords = salinity[:, 0]
    ycoords = salinity[:, 1]
    salinity_values = salinity[:, 2]

    unique_x = np.unique(xcoords)
    unique_y = np.unique(ycoords)
    resolution_x = (xcoords.max() - xcoords.min()) / len(unique_x)
    resolution_y = (ycoords.max() - ycoords.min()) / len(unique_y)

    height = len(unique_y)
    width = len(unique_x)

    transform = from_origin(
        west=xcoords.min(),
        north=ycoords.max(),
        xsize=resolution_x,
        ysize=resolution_y,
    )

    # Same fill array as Food-Security create_salinity_raster
    raster = np.full((height, width + 1), np.nan)
    for i in range(len(salinity_values)):
        row = int((ycoords.max() - ycoords[i]) / resolution_x)
        col = int((xcoords[i] - xcoords.min()) / resolution_y)
        raster[row, col] = salinity_values[i]

    temp_tif = out_tif_path.with_name(out_tif_path.stem + "_temp.tif")
    with rasterio.open(
        temp_tif,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=salinity_values.dtype,
        crs=source_crs,
        transform=transform,
        nodata=nodata_value,
        compress="LZW",
    ) as dst:
        dst.write(raster, 1)

    # Force dtype on COG copy (some GDAL builds otherwise downcast)
    rio_copy(
        temp_tif,
        out_tif_path,
        copy_src_overviews=True,
        driver="COG",
        compress="LZW",
        dtype=str(salinity_values.dtype),
    )
    temp_tif.unlink(missing_ok=True)

    with rasterio.open(out_tif_path) as src:
        if src.dtypes[0] != str(salinity_values.dtype):
            raise TypeError(
                f"Expected COG dtype {salinity_values.dtype}, got {src.dtypes[0]} for {out_tif_path}"
            )


def _mask_nodata(arr, nodata):
    out = arr.astype("float32", copy=True)
    if nodata is not None and not (isinstance(nodata, float) and np.isnan(nodata)):
        out = np.where(out == nodata, np.nan, out)
    out = np.where(out <= -9999, np.nan, out)
    return out


def compute_salinity_increase(salinity_folder, increase_folder, config):
    """Subtract one selected baseline COG from projection (non-historical) salinity COGs.

    Historical years in ``baseline_years`` are never increase targets (no baseline−baseline).
    """
    salinity_folder.mkdir(parents=True, exist_ok=True)
    increase_folder.mkdir(parents=True, exist_ok=True)

    baseline_years = set(config.get("baseline_years", [config["baseline_year"]]))

    baseline_tif = None
    for tif in salinity_folder.glob("*.tif"):
        parsed = matches_criteria(tif, config)
        if not parsed:
            continue
        if (
            parsed["scenario"] == config["baseline_scenario"]
            and parsed["year"] == config["baseline_year"]
        ):
            baseline_tif = tif
            break

    if baseline_tif is None:
        raise FileNotFoundError(
            f"Baseline COG not found in {salinity_folder}. "
            f"Expected scenario={config['baseline_scenario']} and year={config['baseline_year']}."
        )

    with rasterio.open(baseline_tif) as src:
        baseline_data = _mask_nodata(src.read(1), src.nodata)

    rows = []
    errors = []

    for tif in sorted(salinity_folder.glob("*.tif")):
        parsed = matches_criteria(tif, config)
        # Skip all historical/baseline years (not only the selected subtract year)
        if not parsed or parsed["year"] in baseline_years:
            continue

        out_tif = increase_folder / tif.name
        try:
            with rasterio.open(tif) as src:
                data = _mask_nodata(src.read(1), src.nodata)
                profile = src.profile.copy()

            profile.update(dtype="float32", nodata=config["nodata_value"])
            with rasterio.open(out_tif, "w", **profile) as dst:
                dst.write((data - baseline_data).astype("float32"), 1)

            rows.append(
                {"input": str(tif), "baseline": str(baseline_tif), "output": str(out_tif)}
            )
        except Exception as exc:
            errors.append({"file": str(tif), "error": str(exc)})

    return rows, errors


def rename(folder, config):
    """Rename xyz-style COGs to {scenario}_{probability}_{year}.tif."""
    folder.mkdir(parents=True, exist_ok=True)
    rows = []
    errors = []

    for tif in sorted(folder.glob("*.tif")):
        if RENAMED_NAME_RE.match(tif.name):
            continue

        parsed = matches_criteria(tif, config)
        if not parsed:
            continue

        new_name = f"{parsed['scenario']}_{parsed['probability']}_{parsed['year']}.tif"
        new_path = folder / new_name

        try:
            if new_path.exists() and new_path.resolve() != tif.resolve():
                tif.unlink()
            elif new_path != tif:
                tif.rename(new_path)
            rows.append(
                {
                    "old_name": tif.name,
                    "new_name": new_name,
                    "scenario": parsed["scenario"],
                    "year": parsed["year"],
                }
            )
        except Exception as exc:
            errors.append({"file": str(tif), "error": str(exc)})

    return rows, errors


def organize(folder, stac_folder, config, product="salinity"):
    """Move renamed COGs into STAC folders as {probability}_{year}.tif."""
    folder.mkdir(parents=True, exist_ok=True)
    rows = []
    errors = []

    for tif in sorted(folder.glob("*.tif")):
        parsed = matches_criteria(tif, config)
        if not parsed:
            continue

        scenario = parsed["scenario"]
        probability = parsed["probability"]
        year = parsed["year"]

        if not RENAMED_NAME_RE.match(tif.name):
            renamed_path = folder / f"{scenario}_{probability}_{year}.tif"
            if renamed_path.exists():
                continue

        final_name = f"{probability}_{year}.tif"

        if product == "salinity":
            # Historical years of the selected baseline scenario → salinity/baseline/
            if (
                scenario == config["baseline_scenario"]
                and year in config.get("baseline_years", [config["baseline_year"]])
            ):
                dst_folder = stac_folder / "salinity" / "baseline"
            else:
                dst_folder = stac_folder / "salinity" / scenario
        else:
            # Safety: never organize historical years under salinity_increase
            if year in config.get("baseline_years", [config["baseline_year"]]):
                continue
            dst_folder = stac_folder / "salinity_increase" / scenario

        dst_folder.mkdir(parents=True, exist_ok=True)
        dst = dst_folder / final_name

        try:
            shutil.copy2(tif, dst)
            rows.append(
                {
                    "source": str(tif),
                    "scenario": scenario,
                    "year": year,
                    "destination": str(dst),
                }
            )
        except Exception as exc:
            errors.append({"file": str(tif), "error": str(exc)})

    return rows, errors


def sync_metadata_from_cog(metadata_path, sample_cog, *, nodata_json=None):
    """Update NODATA / SPATIAL_RESOLUTION from a sample COG."""
    metadata_path = Path(metadata_path)
    if not metadata_path.is_file():
        print(f"Metadata not found (skip sync): {metadata_path}")
        return None

    with open(metadata_path, encoding="utf-8") as f:
        meta = json.load(f)

    with rasterio.open(sample_cog) as src:
        meta["SPATIAL_RESOLUTION"] = float(abs(src.transform.a))
        meta["NODATA"] = nodata_json

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
        f.write("\n")

    print(f"Updated {metadata_path}")
    print(f"  SPATIAL_RESOLUTION={meta['SPATIAL_RESOLUTION']}")
    print(f"  NODATA={meta['NODATA']}")
    return meta


## 3) Convert selected XYZ files to salinity COGs

COGs keep the **same name as the source xyz** (only extension changes).


In [4]:
conversion_rows = []
conversion_errors = []
skipped_files = []

for xyz_path in sorted(config["source_dir"].glob("*.xyz")):
    parsed = matches_criteria(xyz_path, config)
    if not parsed:
        skipped_files.append(str(xyz_path))
        continue

    out_tif = salinity_staging / xyz_path.with_suffix(".tif").name

    try:
        convert_xyz_to_cog(xyz_path, out_tif, config)
        with rasterio.open(out_tif) as src:
            cog_dtype = src.dtypes[0]
        conversion_rows.append({
            "xyz": str(xyz_path),
            "scenario": parsed["scenario"],
            "year": parsed["year"],
            "cog": str(out_tif),
            "dtype": cog_dtype,
        })
    except Exception as exc:
        conversion_errors.append({"xyz": str(xyz_path), "error": str(exc)})

print(f"Selected and converted: {len(conversion_rows)}")
print(f"Skipped (did not match criteria): {len(skipped_files)}")
print(f"Conversion errors: {len(conversion_errors)}")

if conversion_rows:
    df_conv = pd.DataFrame(conversion_rows).sort_values(["scenario", "year"])
    display(df_conv)
    bad = df_conv[df_conv["dtype"] != "float64"]
    if len(bad):
        raise TypeError(
            "Some COGs are not float64 (Food-Security dtype). "
            "Re-run helpers (§2) then this cell after deleting cogs_staging/salinity. "
            f"Bad rows:\n{bad}"
        )
    print("All converted COGs are float64 — OK")
if conversion_errors:
    display(pd.DataFrame(conversion_errors))


Selected and converted: 19
Skipped (did not match criteria): 15
Conversion errors: 0


,xyz,scenario,year,cog,dtype
0,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2rb1y,2030,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,float64
1,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2rb1y,2040,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,float64
2,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2rb1y,2050,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,float64
3,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2y,2030,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,float64
4,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2y,2040,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,float64
5,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2y,2050,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,float64
6,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45y,2014,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,float64
7,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45y,2015,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,float64
8,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45y,2016,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,float64
9,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45y,2018,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,float64


All converted COGs are float64 — OK


## 4) Compute salinity increase

Subtracts the single selected baseline (`baseline_scenario` + `baseline_year`) from **projection** COGs only.

Years in `baseline_years` are skipped (no historical−historical / baseline−baseline).


In [5]:
increase_rows, increase_errors = compute_salinity_increase(
    salinity_staging,
    salinity_increase_staging,
    config,
)

print(f"Salinity increase files created: {len(increase_rows)}")
print(f"Errors: {len(increase_errors)}")

if increase_rows:
    display(pd.DataFrame(increase_rows))
if increase_errors:
    display(pd.DataFrame(increase_errors))


Salinity increase files created: 15
Errors: 0


,input,baseline,output
0,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
1,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
2,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
3,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
4,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
5,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
6,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
7,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
8,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
9,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...


## 5) Rename COGs in staging folders

Renaming rule:

`{scenario}_{probability}_{year}.tif`

This step runs **before** organization.


In [6]:
salinity_rename_rows, salinity_rename_errors = rename(salinity_staging, config)
increase_rename_rows, increase_rename_errors = rename(salinity_increase_staging, config)

print(f"Salinity renamed: {len(salinity_rename_rows)}")
print(f"Salinity increase renamed: {len(increase_rename_rows)}")

if salinity_rename_rows:
    display(pd.DataFrame(salinity_rename_rows))
if increase_rename_rows:
    display(pd.DataFrame(increase_rename_rows))

all_rename_errors = salinity_rename_errors + increase_rename_errors
if all_rename_errors:
    display(pd.DataFrame(all_rename_errors))


Salinity renamed: 19
Salinity increase renamed: 15


,old_name,new_name,scenario,year
0,P50_cc45sm2rb1y30.tif,cc45sm2rb1y_p50_2030.tif,cc45sm2rb1y,2030
1,P50_cc45sm2rb1y40.tif,cc45sm2rb1y_p50_2040.tif,cc45sm2rb1y,2040
2,P50_cc45sm2rb1y50.tif,cc45sm2rb1y_p50_2050.tif,cc45sm2rb1y,2050
3,P50_cc45sm2y30.tif,cc45sm2y_p50_2030.tif,cc45sm2y,2030
4,P50_cc45sm2y40.tif,cc45sm2y_p50_2040.tif,cc45sm2y,2040
5,P50_cc45sm2y50.tif,cc45sm2y_p50_2050.tif,cc45sm2y,2050
6,P50_cc45y14.tif,cc45y_p50_2014.tif,cc45y,2014
7,P50_cc45y15.tif,cc45y_p50_2015.tif,cc45y,2015
8,P50_cc45y16.tif,cc45y_p50_2016.tif,cc45y,2016
9,P50_cc45y18.tif,cc45y_p50_2018.tif,cc45y,2018


,old_name,new_name,scenario,year
0,P50_cc45sm2rb1y30.tif,cc45sm2rb1y_p50_2030.tif,cc45sm2rb1y,2030
1,P50_cc45sm2rb1y40.tif,cc45sm2rb1y_p50_2040.tif,cc45sm2rb1y,2040
2,P50_cc45sm2rb1y50.tif,cc45sm2rb1y_p50_2050.tif,cc45sm2rb1y,2050
3,P50_cc45sm2y30.tif,cc45sm2y_p50_2030.tif,cc45sm2y,2030
4,P50_cc45sm2y40.tif,cc45sm2y_p50_2040.tif,cc45sm2y,2040
5,P50_cc45sm2y50.tif,cc45sm2y_p50_2050.tif,cc45sm2y,2050
6,P50_cc45y30.tif,cc45y_p50_2030.tif,cc45y,2030
7,P50_cc45y40.tif,cc45y_p50_2040.tif,cc45y,2040
8,P50_cc45y50.tif,cc45y_p50_2050.tif,cc45y,2050
9,P50_cc85sb2y30.tif,cc85sb2y_p50_2030.tif,cc85sb2y,2030


## 6) Organize into final folder structure

Final filename in STAC folders:

`{probability}_{year}.tif`

- `salinity/baseline/` — `baseline_scenario` × years in `baseline_years` (e.g. 2014–2018)
- `salinity/{scenario}/` — projection scenarios/years
- `salinity_increase/{scenario}/` — increase for projection years only

Files are saved in `stac_folder/` (same level as `cogs_staging`):

- `stac_folder/salinity/baseline/`
- `stac_folder/salinity/{scenario}/`
- `stac_folder/salinity_increase/{scenario}/`


In [7]:
salinity_org_rows, salinity_org_errors = organize(salinity_staging, stac_folder, config, product="salinity")
increase_org_rows, increase_org_errors = organize(salinity_increase_staging, stac_folder, config, product="salinity_increase")

print(f"Salinity organized: {len(salinity_org_rows)}")
print(f"Salinity increase organized: {len(increase_org_rows)}")

if salinity_org_rows:
    display(pd.DataFrame(salinity_org_rows).sort_values(["scenario", "year"]))
if increase_org_rows:
    display(pd.DataFrame(increase_org_rows).sort_values(["scenario", "year"]))

all_org_errors = salinity_org_errors + increase_org_errors
if all_org_errors:
    display(pd.DataFrame(all_org_errors))


Salinity organized: 19
Salinity increase organized: 15


,source,scenario,year,destination
0,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2rb1y,2030,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
1,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2rb1y,2040,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
2,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2rb1y,2050,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
3,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2y,2030,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
4,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2y,2040,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
5,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2y,2050,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
6,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45y,2014,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
7,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45y,2015,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
8,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45y,2016,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
9,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45y,2018,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...


,source,scenario,year,destination
0,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2rb1y,2030,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
1,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2rb1y,2040,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
2,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2rb1y,2050,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
3,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2y,2030,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
4,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2y,2040,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
5,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45sm2y,2050,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
6,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45y,2030,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
7,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45y,2040,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
8,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc45y,2050,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...
9,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...,cc85sb2y,2030,P:\11211454-002-idt\IDP\Vietnam\Mekong\salinit...


## 7) Sync metadata JSON with new COG grid / nodata

Updates `NODATA` (JSON `null` for NaN) and `SPATIAL_RESOLUTION` from a sample COG so `11_salinity.ipynb` stays consistent.


In [8]:
# Sync STAC metadata NODATA / resolution from a sample COG (NaN -> JSON null)
sample_cogs = sorted((stac_folder / "salinity").rglob("*.tif"))
if not sample_cogs:
    raise FileNotFoundError(f"No salinity COGs under {stac_folder / 'salinity'}")

sample = sample_cogs[0]
print("Sample COG:", sample)

for product in ("salinity", "salinity_increase"):
    meta_path = stac_folder / product / f"metadata_{product}.json"
    sync_metadata_from_cog(meta_path, sample, nodata_json=None)


Sample COG: P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\salinity\baseline\p50_2014.tif
Updated P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\salinity\metadata_salinity.json
  SPATIAL_RESOLUTION=1992.7272727272727
  NODATA=None
Updated P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\salinity_increase\metadata_salinity_increase.json
  SPATIAL_RESOLUTION=1992.7272727272727
  NODATA=None
